# Neural Networks: Layers, Towers, and Learnable Transforms


> **tldr:** NeuralGCM builds learnable transformations across spatial grids using three distinct abstraction tiers: numerical Layers, coordinate-aware Towers, and dictionary Transforms. Each level provides flexible operations at a distinct level of granularity, all of which will be covered in this tutorial through step-by-step examples.

**Key concepts:**
*   **Layers (Raw Arrays):** Foundation neural network modules (MLPs, CNNs) that operate directly on standard numerical arrays across explicit positional axes
*   **Towers (`coordax.Field`):** Bridging wrappers that lift layers to operate on labeled fields, automatically managing spatial vectorization and weight sharing
*   **Learned Transforms (`dict` mapping):** Modules that define input/output transformations to extend towers to a `Transform` protocol (`dict[str, cx.Field] -> dict[str, cx.Field]`)


In [ ]:
import coordax as cx
from flax import nnx
import jax.numpy as jnp
from neuralgcm.experimental.core import standard_layers
from neuralgcm.experimental.core import towers
from neuralgcm.experimental.core import learned_transforms
from neuralgcm.experimental.core import coordinates
from neuralgcm.experimental.core import transforms
from neuralgcm.experimental.core import feature_transforms
from neuralgcm.experimental.core import spherical_harmonics

## 1. Networks and Layers

**Use-case:** Define low-level neural network layers that operate directly on raw positional arrays

At this level, operations focus purely on tensor slice mechanics. Each layer enforces an explicit data layout contract governed by `input_size` and `output_size` parameters. While standard linear layers and MLPs transform a single channel dimension, spatial layers like 2D convolutions process multiple grid axes simultaneously.

Towers (discussed in Section 2) leverage these foundational layer modules by wrapping them to automatically manage batch vectorization and coordinate mappings across unlisted dimensions.

Here are examples of layers available in `standard_layers.py`.

### 1.1 Multi-Layer Perceptrons (MLP)

The `{py:class}MlpUniform` class defines a multi-layer perceptron with uniform hidden layer sizes. When instantiating this layer, `input_size` specifies the exact number of input features, and `output_size` determines the number of output features.

Note: layer modules do not explicitly deal with unbatched inputs, as batch vectorization is handled automatically by Tower modules

In [ ]:
# Define dimensions
input_size = 10
output_size = 5
hidden_size = 20
n_hidden_layers = 2

# Initialize the MLP
mlp = standard_layers.MlpUniform(
    input_size=input_size,
    output_size=output_size,
    hidden_size=hidden_size,
    n_hidden_layers=n_hidden_layers,
    rngs=nnx.Rngs(0),
)

# Create a simple input array without a batch dimension
x = jnp.ones((input_size,))

# Apply the MLP
y = mlp(x)

# Inspect the output shape
print(f'Input shape: {x.shape}')
print(f'Output shape: {y.shape}')


### 1.2 Spatial Layers with `ConvLonLat`

`{py:class}ConvLonLat` defines 2D convolutional layer on data with longitude and latitude dimensions.

This layer operates on several dimensions with an assumed spatial structure. For example, passing an input of shape `(12, 32, 64)` (12 features on a 32x64 grid) produces an output of shape `(6, 32, 64)` (6 features on the same grid).

In [ ]:
# Define dimensions
input_size = 12
output_size = 6
hidden_size = 24
hidden_layers = 2
lon_size = 32
lat_size = 64

# Build the CnnLonLat network
cnn = standard_layers.CnnLonLat.build_using_factories(
    input_size=input_size,
    output_size=output_size,
    hidden_size=hidden_size,
    hidden_layers=hidden_layers,
    kernel_size=(3, 3),
    rngs=nnx.Rngs(0),
)

# Create a sample input with spatial dimensions
x = jnp.ones((input_size, lon_size, lat_size))

# Apply the network
y = cnn(x)

# Inspect shapes
print(f'Input shape: {x.shape}')
print(f'Output shape: {y.shape}')


### 1.3 Note on `build_using_factories`

Many layers/towers implement `build_using_factories` class methods to delegate sub-module construction to the containing class. This allows passing factory functions (often configured using `{py:func}functools.partial`) instead of fully constructed objects, enabling late binding of shapes or other configuration parameters.

## 2. Towers: Bridging Layers and Fields

**Use-case:** Lift layers processing arrays to `{py:class}coordax.Field` transforms with automatic vectorization.

Towers wrap raw neural network layers to interface cleanly with `{py:class}coordax.Field` objects. By defining `inputs_in_dims` (active input axes) and `out_dims` (target output axes), the Tower establishes an automatic broadcasting contract. Any unlisted dimensions in the input field—such as batches or spatial grid dimensions—are automatically vectorized across using `{py:func}coordax.cmap`.
### Example: Predicting Atmospheric State
We examine this workflow by modeling atmospheric states across a 64x32 spatial grid. The setup maps 64 input feature channels to 13 target vertical pressure levels labeled via `{py:meth}coordinates.PressureLevels.with_13_era5_levels()`.
We construct and evaluate two canonical tower configurations:
1.  **Column Tower**: Applies an MLP along feature columns, operating point-by-point across the grid.
2.  **Spatial Tower**: Applies 2D convolutional layers across spatial horizontal grid dimensions.
Finally, we add an unlisted batch dimension of size 4 to demonstrate automatic vectorization.

In [ ]:
# Setup Grid and Inputs
grid = coordinates.LonLatGrid.TL31()
print(f'Grid shape: {grid.shape}')  # Expected to be (64, 32)

# Create random inputs: 64 features on the grid
input_size = 64

# Use PressureLevels for output coordinate
output_coords = coordinates.PressureLevels.with_13_era5_levels()
output_size = output_coords.shape[0]
print(f'Output size: {output_size}')

# Create a raw array
raw_array = jnp.ones((input_size,) + grid.shape)

# Create a coordax Field
# We tag the first dimension as 'channel' and pass the grid for spatial dimensions
input_field = cx.field(raw_array, 'channel', grid)

print(f'Input field shape: {input_field.shape}')
print(f'Input field dims: {input_field.dims}')


### 2.1 Column Tower Example
A column tower applies an underlying multi-layer perceptron (MLP) across feature channels independently at each spatial grid point. By configuring `inputs_in_dims=('channel',)` and `out_dims=(output_coords,)`, the tower explicitly identifies the active transformation axis while automatically vectorizing over the unlisted `longitude` and `latitude` spatial dimensions.

In [ ]:
import functools

# Define MLP factory using functools.partial
mlp_factory = functools.partial(
    standard_layers.MlpUniform,
    hidden_size=32,
    n_hidden_layers=2,
)

# Build the ForwardTower
column_tower = towers.ForwardTower.build_using_factories(
    input_size=input_size,
    output_size=output_size,
    inputs_in_dims=('channel',),
    out_dims=(output_coords,),
    neural_net_factory=mlp_factory,
    rngs=nnx.Rngs(0),
)

# Apply the tower to the input field
output_field = column_tower(input_field)

print(f'Output field shape: {output_field.shape}')
print(f'Output field dims: {output_field.dims}')


### 2.2 Spatial Tower Example
In contrast to column towers, a spatial tower processes regional structural neighborhoods by wrapping a 2D spatial convolution (`CnnLonLat`). The configuration specifies that the internal network operates simultaneously on feature channels and spatial coordinates via `inputs_in_dims=('channel', grid)` and `out_dims=(output_coords, grid)`, where `grid` automatically expands to represent both `longitude` and `latitude`.

In [ ]:
# Define CNN factory using functools.partial
cnn_factory = functools.partial(
    standard_layers.CnnLonLat.build_using_factories,
    hidden_size=16,
    hidden_layers=1,
    kernel_size=(3, 3),
)

# Build the ForwardTower
spatial_tower = towers.ForwardTower.build_using_factories(
    input_size=input_size,
    output_size=output_size,
    inputs_in_dims=('channel', grid),
    out_dims=(output_coords, grid),
    neural_net_factory=cnn_factory,
    rngs=nnx.Rngs(0),
)

# Apply the tower to the input field
output_spatial_field = spatial_tower(input_field)

print(f'Output spatial field shape: {output_spatial_field.shape}')
print(f'Output spatial field dims: {output_spatial_field.dims}')


### 2.3 Vectorization: Automatic Batching

Towers automatically handle dimensions not specified in `inputs_in_dims`. To demonstrate this, let's add a `batch` dimension of size 4 to inputs and observe that the tower automatically vectorizes of it.

In [ ]:
# Create batched inputs
batch_size = 4
batched_raw_array = jnp.ones((batch_size, input_size) + grid.shape)

# Create a field with a 'batch' dimension
batched_input_field = cx.field(batched_raw_array, 'batch', 'channel', grid)

print(f'Batched input field shape: {batched_input_field.shape}')
print(f'Batched input field dims: {batched_input_field.dims}')

# Apply the column tower (which only operates on 'channel')
# It will automatically vectorize over 'batch', 'longitude', and 'latitude'
batched_output_column = column_tower(batched_input_field)
print(f'Column tower output shape: {batched_output_column.shape}')
print(f'Column tower output dims: {batched_output_column.dims}')

# Apply the spatial tower (which operates on 'channel' and grid)
# It will automatically vectorize over 'batch'
batched_output_spatial = spatial_tower(batched_input_field)
print(f'Spatial tower output shape: {batched_output_spatial.shape}')
print(f'Spatial tower output dims: {batched_output_spatial.dims}')


### 2.4 Advanced Tower Variations

Specialized towers handle more complex scenarios:

*   **`RecurrentTower`**: Applies an RNN cell (LSTM, GRU) to inputs and a carry state. Useful for sequential data or maintaining state across time steps.
*   **`TransformerTower`**: Wraps transformer networks. Supports optional `latents` and `mask` inputs for decoding or attention masking.

Both follow the principle of specifying dimensions to operate on and dimensions to vectorize over.

Towers also support `apply_remat=True` for gradient checkpointing to save memory during training.

See [towers_test.py](file:///google/src/cloud/dkochkov/ngcm_notebooks/google3/third_party/py/neuralgcm/experimental/core/towers_test.py) for more examples.

## 3. Learnable Transforms

**Use-case:** Map dictionaries of fields to dictionaries of fields (`dict[str, Field] -> dict[str, Field]`) using neural network towers.

Learnable transforms serve as the final tier in the abstraction hierarchy, embedding neural network towers directly into the standard `Transform` protocol. A canonical implementation is `{py:class}ForwardTowerTransform`, which orchestrates the bidirectional flow of data between dictionary mappings and the internal forward tower.

### Execution Flow

When evaluated, `{py:class}ForwardTowerTransform` processes input field dictionaries through a structured pipeline:
1.  **Input Filtering:** Executes an optional `inputs_transform` to pre-process or select relevant incoming fields.
2.  **Feature Stacking:** Concatenates selected input fields along `concat_dims` into a unified single field tensor.
3.  **Network Evaluation:** Evaluates the internal wrapper **Tower** over the concatenated field.
4.  **Target Partitioning:** Splits the transformed output field back into separate output target fields according to `target_split_axes`.
5.  **Final Transformation:** Runs an optional `out_transform` for post-processing.

### Implementation constraints
*   **Leading channel axis**: Currently the leading dimension processed by the tower (i.e. first dim in tower's `inputs_in_dims` and `out_dims`) must correspond to input/output channels. This simplifies axes manipulation, but will be relaxed in the future updates.

Below, we demonstrate this pipeline with a conceptual example.


In [ ]:
# Define target split axes
# Total output size must be 13 to match our tower setup
target_split_axes = {
    'temp_lower': cx.SizedAxis('level', 5),
    'temp_upper': cx.SizedAxis('level', 8),
}

# We can reuse the mlp_factory from earlier

# For ForwardTowerTransform, we pass a factory for the TOWER,
# which in turn uses a factory for the network.
# Here we use a partial to pre-configure the tower factory.
tower_factory = functools.partial(
    towers.ForwardTower.build_using_factories,
    inputs_in_dims=('channel',),
    out_dims=('level',),
    neural_net_factory=mlp_factory,
)

# We need to provide representative input shapes
input_shapes = {'features': input_field}

# Build the ForwardTowerTransform
# We specify concat_dims=('channel',) to indicate that 'channel' is the feature dimension
transform = learned_transforms.ForwardTowerTransform.build_using_factories(
    input_shapes=input_shapes,
    target_split_axes=target_split_axes,
    tower_factory=tower_factory,
    concat_dims=('channel',),
    rngs=nnx.Rngs(0),
)

# Create dummy input dictionary
inputs_dict = {'features': input_field}

# Apply the transform
output_dict = transform(inputs_dict)

print('Output keys:', output_dict.keys())
print('Shape of "temp_lower":', output_dict['temp_lower'].shape)
print('Shape of "temp_upper":', output_dict['temp_upper'].shape)


## 4. Full Example: Setting up a Learned Transform

**Use-case:** Combine features, normalization, and a learnable transform into a complete workflow.

Here we demonstrate a complete process of setting up a Learnable Transform with expanded features computed from the input state, followed by a normalization demonstrating typical ML worflow.

### Scenario
We assume access to inputs containing 2 fields: `u_component_of_wind` and `v_component_of_wind` (13 pressure levels on a 64x32 grid).

Transform components:
1.  **Prognostics**: Original `u` and `v` fields (Identity)
2.  **Div/Curl**: Divergence and vorticity computed from `u` and `v`
3.  Use `Merge` to construct all input features
4.  Apply normalization (`StreamNorm`) transform sequentially after merge
5.  Pass the result to the `ForwardTowerTransform` that we will configure to predict '10m_wind_speed' and 'surface_drag' (lon-lat dimensions only)

We will make use of `output_shapes` and `build_using_factories` methods to avoid calculating required shapes manually.

In [ ]:
# Setup inputs
grid = coordinates.LonLatGrid.TL31()
levels = coordinates.PressureLevels.with_13_era5_levels()

# Create dummy inputs with shape (13, 64, 32)
u_array = jnp.ones(levels.shape + grid.shape)
v_array = jnp.ones(levels.shape + grid.shape)
inputs = {
    'u_component_of_wind': cx.field(u_array, levels, grid),
    'v_component_of_wind': cx.field(v_array, levels, grid),
}

# 1-2-3. Define feature transforms
raw_features = transforms.Merge({
    'prognostics': transforms.Identity(),
    'div_curl': transforms.VelocityToDivCurl(spherical_harmonics.YlmMapper()),
})

# Use output_shapes to avoid manual calculation
feature_struct = raw_features.output_shapes(inputs)
print('Merged feature keys:', feature_struct.keys())

# 4. Setup Normalization
norm_transform = transforms.StreamNorm.for_inputs_struct(
    inputs_struct=feature_struct,
    independent_axes=(),  # scalar normalization
    skip_unspecified=True,
)
inputs_transform = transforms.Sequential([raw_features, norm_transform])
# inputs_transforms.output_shapes will be used to infer tower's input_size arg.

# 5. Setup Main Transform
main_mlp_factory = functools.partial(
    standard_layers.MlpUniform, hidden_size=64, n_hidden_layers=2
)
main_tower_factory = functools.partial(
    towers.ForwardTower.build_using_factories,
    inputs_in_dims=(),  # operates on concatenated features
    out_dims=('output_channel',),
    neural_net_factory=main_mlp_factory,
)

# Predict fields with only lon-lat dimensions
target_split_axes = {  # will specify output_size == 2 when building the tower.
    '10m_wind_speed': cx.Scalar(),
    'surface_drag': cx.Scalar(),
}

main_transform = learned_transforms.ForwardTowerTransform.build_using_factories(
    input_shapes=inputs,
    target_split_axes=target_split_axes,
    tower_factory=main_tower_factory,
    inputs_transform=inputs_transform,
    concat_dims=(levels,),
    rngs=nnx.Rngs(1),
)

# Apply the full transform
outputs = main_transform(inputs)

print('\nFinal Output keys:', outputs.keys())
print('Shape of "10m_wind_speed":', outputs['10m_wind_speed'].shape)
print('Shape of "surface_drag":', outputs['surface_drag'].shape)


### Note on Design Choices

*   **`output_shapes`**: We used `raw_features.output_shapes(inputs)` to get the structure and shapes of the generated features without actually computing them. This is useful for setting up subsequent transforms like `StreamNorm` that need to know the coordinates of the fields they operate on.
*   **`build_using_factories`**: We used this pattern for both the nested `surf_transform` and the `main_transform`. This allowed us to avoid calculating the total number of concatenated features by hand; the `build_using_factories` method evaluates the shapes internally and instantiates the network with the correct sizes.
*   **Feature Choices**: Prognostics and div/curl are standard features for atmospheric models. We could have added `LatitudeFeatures` to provide positional context or learned surface features to let the transform learn location-specific features.

### Advanced Transform Variations

Similar to Towers, there are advanced variations of learned transforms:
*   **`RecurrentTowerTransform`**: Combines `RecurrentTower` with the dictionary-to-dictionary mapping, handling state keys
*   **`TransformerTowerTransform`**: Combines `TransformerTower` for handling inputs, latents, and masks in a transform setting
*   **`LandSeaIceTowersTransform`**: A specialized transform that combines different towers for land, sea, and sea-ice regions and blends their outputs based on masks